In [7]:
!pip install -q chromadb langchain langchain-chroma langchain-huggingface langchain-openai sentence-transformers fastparquet

In [13]:
import pandas as pd
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

DATA_PATH = "/content/mirkvartir_moscow_flats_10000.parquet"
DB_DIR = "./chroma_flats_db"

EMB_MODEL_NAME = "cointegrated/rubert-tiny2"

print("Загружаем данные...")
df = pd.read_parquet(DATA_PATH)

# Оставляем только те, где есть описание
df = df.dropna(subset=['description_full', 'flat_id', 'price_total', 'url'])

df_subset = df

texts = df_subset['description_full'].tolist()
ids = df_subset['flat_id'].astype(str).tolist()

# Формируем метаданные (то, что приклеится к тексту в базе)
metadatas = [
    {
        "flat_id": str(row['flat_id']),
        "price_mln": round(row['price_total'] / 1_000_000, 2),
        "url": str(row['url'])
    }
    for _, row in df_subset.iterrows()
]


Загружаем данные...


In [12]:
# СОЗДАНИЕ ВЕКТОРНОЙ БАЗЫ (ChromaDB)
print(f"Загружаем модель эмбеддингов: {EMB_MODEL_NAME}...")
embeddings = HuggingFaceEmbeddings(model_name=EMB_MODEL_NAME)

print(f"Создаем векторную базу на {len(texts)} квартир")
vectorstore = Chroma.from_texts(
    texts=texts,
    embedding=embeddings,
    metadatas=metadatas,
    ids=ids,
    persist_directory=DB_DIR,
    collection_name="moscow_flats"
)
print(f"База сохранена в папку {DB_DIR}")

# ФУНКЦИЯ ПОИСКА АЛЬТЕРНАТИВ
def find_similar_flats(query_text: str, k: int = 3):
    """Ищет похожие квартиры по смыслу текста"""
    print(f"\n Ищем конкурентов для запроса...")
    results = vectorstore.similarity_search(query_text, k=k)

    for i, doc in enumerate(results, 1):
        meta = doc.metadata
        print(f"--- Конкурент {i} ---")
        print(f"Цена: {meta['price_mln']} млн руб. | Ссылка: {meta['url']}")
        print(f"Текст: {doc.page_content[:150]}...\n")

    return results

Загружаем модель эмбеддингов: cointegrated/rubert-tiny2...


Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Создаем векторную базу на 9280 квартир
База сохранена в папку ./chroma_flats_db


In [14]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from google.colab import userdata

# НАСТРОЙКА LLM (PROXYAPI)
os.environ["OPENAI_API_KEY"] = userdata.get('PROXYAPI')
os.environ["OPENAI_API_BASE"] = "https://api.proxyapi.ru/openai/v1"

# Инициализируем модель
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)

# ПРОМПТ ДЛЯ RAG
RAG_PROMPT = ChatPromptTemplate.from_template("""\
Ты — профессиональный ИИ-аналитик элитной и массовой недвижимости.
Твоя задача — оценить, насколько выгодна сделка для клиента.
Отвечай ТОЛЬКО на основе предоставленного контекста (найденных конкурентов).
Если конкурентов нет, скажи об этом честно.

ЦЕЛЕВАЯ КВАРТИРА КЛИЕНТА:
Цена: {target_price} млн руб.
Описание: {target_desc}

НАЙДЕННЫЕ КОНКУРЕНТЫ НА РЫНКЕ (КОНТЕКСТ):
{context}

НАПИШИ КРАТКИЙ ОТЧЕТ:
1. Оценка цены: Насколько цена клиента адекватна на фоне конкурентов?
2. Плюсы и минусы: Что есть у конкурентов, чего нет у клиента (и наоборот)?
3. Вердикт: Стоит ли брать эту квартиру или конкуренты лучше?
""")

# ФУНКЦИЯ-ГЕНЕРАТОР ОТЧЕТА
def generate_report(target_desc: str, target_price_mln: float):
    print(f"Ищем конкурентов для квартиры за {target_price_mln} млн руб...")

    # Ищем квартиры только в диапазоне +/- 30% от цены
    min_price = target_price_mln * 0.7
    max_price = target_price_mln * 1.3

    results = vectorstore.similarity_search(
        query=target_desc,
        k=3,
        filter={
            "$and": [
                {"price_mln": {"$gte": min_price}},
                {"price_mln": {"$lte": max_price}}
            ]
        }
    )

    if not results:
        return "К сожалению, в базе нет похожих квартир в этом ценовом диапазоне для сравнения."

    # Формируем текст контекста для LLM
    context_parts = []
    for i, doc in enumerate(results, 1):
        price = doc.metadata['price_mln']
        url = doc.metadata['url']
        text = doc.page_content[:500]
        context_parts.append(f"--- Конкурент {i} ---\nЦена: {price} млн\nСсылка: {url}\nОписание: {text}...")

    context_str = "\n\n".join(context_parts)

    print("Отправляем данные в LLM для генерации отчета...\n")

    # Собираем промпт и вызываем модель
    prompt_value = RAG_PROMPT.invoke({
        "target_price": target_price_mln,
        "target_desc": target_desc,
        "context": context_str
    })

    response = llm.invoke(prompt_value)

    print("НАЙДЕННЫЕ КОНКУРЕНТЫ ДЛЯ СРАВНЕНИЯ:")
    for doc in results:
        print(f"- {doc.metadata['price_mln']} млн руб. | {doc.metadata['url']}")

    print("\n" + "="*60)
    print("АНАЛИТИЧЕСКИЙ ИИ-ОТЧЕТ")
    print("="*60)

    return response.content


In [15]:
# ТЕСТ
my_flat_desc = "Продается светлая уютная студия с новым евроремонтом 20 кв. м, в центре москвы. Рядом парк"
my_flat_price = 25 # Адекватная цена для студии

report = generate_report(my_flat_desc, my_flat_price)
print(report)

Ищем конкурентов для квартиры за 25 млн руб...
Отправляем данные в LLM для генерации отчета...

НАЙДЕННЫЕ КОНКУРЕНТЫ ДЛЯ СРАВНЕНИЯ:
- 23.95 млн руб. | https://www.mirkvartir.ru/358746081/
- 27.0 млн руб. | https://www.mirkvartir.ru/356074583/
- 20.5 млн руб. | https://www.mirkvartir.ru/357859921/

АНАЛИТИЧЕСКИЙ ИИ-ОТЧЕТ
### Отчет по оценке целевой квартиры клиента

1. **Оценка цены**: 
   - Целевая квартира клиента стоит 25 млн руб. 
   - Конкуренты предлагают квартиры по следующим ценам: 
     - Конкурент 1: 23.95 млн руб. (студия в зеленом районе, близко к метро)
     - Конкурент 2: 27.0 млн руб. (трехкомнатная квартира в центре, рядом с парком)
     - Конкурент 3: 20.5 млн руб. (двухкомнатная квартира, хорошая локация, недалеко от метро)
   - Цена клиента находится в диапазоне конкурентов, но выше самой дешевой квартиры (20.5 млн руб.) и ниже самой дорогой (27.0 млн руб.). В целом, цена клиента адекватна, но есть более выгодные предложения.

2. **Плюсы и минусы**:
   - **Плюсы целев

In [16]:
!zip -r chroma_flats_db.zip chroma_flats_db/

  adding: chroma_flats_db/ (stored 0%)
  adding: chroma_flats_db/chroma.sqlite3 (deflated 50%)
  adding: chroma_flats_db/cddf8e21-018c-45d9-a9bd-1372f0c8a328/ (stored 0%)
  adding: chroma_flats_db/cddf8e21-018c-45d9-a9bd-1372f0c8a328/header.bin (deflated 57%)
  adding: chroma_flats_db/cddf8e21-018c-45d9-a9bd-1372f0c8a328/link_lists.bin (deflated 78%)
  adding: chroma_flats_db/cddf8e21-018c-45d9-a9bd-1372f0c8a328/data_level0.bin (deflated 12%)
  adding: chroma_flats_db/cddf8e21-018c-45d9-a9bd-1372f0c8a328/index_metadata.pickle (deflated 65%)
  adding: chroma_flats_db/cddf8e21-018c-45d9-a9bd-1372f0c8a328/length.bin (deflated 82%)
